# Notebook 03 – Statistical Validation

Validate whether relationships between user context (personality, mood, persona, device, shopping behaviour) and UI preferences are statistically significant.

Only statistically supported relationships are forwarded to later notebooks.


## Expected Outputs
- `reports/Statistical_Validation/tables/statistical_results.xlsx` (all tests retained)
- `reports/Statistical_Validation/tables/strong_evidence.xlsx`
- `reports/Statistical_Validation/tables/moderate_evidence.xlsx`
- `reports/Statistical_Validation/tables/exploratory_evidence.xlsx`
- `reports/Statistical_Validation/tables/top_evidence.xlsx`
- `reports/Statistical_Validation/statistical_summary.md`
- Heatmaps and figures under `reports/Statistical_Validation/`

> **Run the setup cell below first.** All relationships are kept; evidence tiers classify rather than discard results.


In [8]:
# --- Standard library ---
import logging
import sys
from pathlib import Path

# --- Third-party ---
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


def _bootstrap_project() -> Path:
    search_from = Path.cwd().resolve()
    candidates = [search_from, *search_from.parents]
    nested_root = search_from / "EvidenceBasedAdaptiveUI"
    if (nested_root / "src" / "config.py").exists():
        candidates.insert(0, nested_root)

    for candidate in candidates:
        if (candidate / "src" / "config.py").exists():
            root = str(candidate)
            if root not in sys.path:
                sys.path.insert(0, root)
            return candidate

    raise FileNotFoundError(
        "Could not find project root containing src/config.py."
    )


_bootstrap_project()

# --- Project imports ---
from src.preprocessing.columns import (
    ALL_UI_COLUMNS,
    BIG_FIVE_LEVEL_COLUMNS,
    CONTEXT_FEATURE_COLUMNS,
    DESKTOP_UI_COLUMNS,
    GLOBAL_UI_COLUMNS,
    MOBILE_UI_COLUMNS,
    get_statistical_predictor_columns,
)
from src.statistics.reporting import generate_statistical_summary
from src.statistics.evidence import (
    export_statistical_outputs,
    summarize_evidence_statistics,
)
from src.statistics.validator import run_pairwise_validation
from src.utils.notebook import setup_notebook
from src.visualization.statistical_plots import (
    plot_association_heatmap,
    plot_top_relationships,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["figure.figsize"] = (10, 6)

PATHS, REPORTS = setup_notebook("Statistical_Validation")
TABLES = REPORTS / "tables"
FIGURES = REPORTS / "figures"
HEATMAPS = REPORTS / "heatmaps"
for folder in (TABLES, FIGURES, HEATMAPS):
    folder.mkdir(parents=True, exist_ok=True)

dataset_path = PATHS.data_processed / "clean_dataset.csv"
df = pd.read_csv(dataset_path)
logger.info("Loaded clean dataset: %s rows, %s columns", *df.shape)

# Context (mood, persona, device) + Big Five levels — 8 predictors × 41 UI elements
predictors = [
    column for column in get_statistical_predictor_columns() if column in df.columns
]
ui_elements = [column for column in ALL_UI_COLUMNS if column in df.columns]

print("Context predictors:", [c for c in predictors if c in CONTEXT_FEATURE_COLUMNS])
print("Big Five levels:", [c for c in predictors if c in BIG_FIVE_LEVEL_COLUMNS])
print(f"Global UI: {len(GLOBAL_UI_COLUMNS)} | Desktop: {len(DESKTOP_UI_COLUMNS)} | Mobile: {len(MOBILE_UI_COLUMNS)}")
print(f"Predictors: {len(predictors)} | UI elements: {len(ui_elements)} | Total pairs: {len(predictors) * len(ui_elements)}")
display(df[predictors + ui_elements[:3]].head())




INFO: Loaded clean dataset: 200 rows, 79 columns


Context predictors: ['primary_persona', 'current_mood', 'primary_device']
Big Five levels: ['Extraversion_Level', 'Agreeableness_Level', 'Conscientiousness_Level', 'Neuroticism_Level', 'Openness_Level']
Global UI: 14 | Desktop: 13 | Mobile: 14
Predictors: 8 | UI elements: 41 | Total pairs: 328


,primary_persona,current_mood,primary_device,Extraversion_Level,Agreeableness_Level,Conscientiousness_Level,Neuroticism_Level,Openness_Level,font_style_pref,font_size_pref,color_theme_pref
0,The Impulsive Buyer (I make quick decisions ba...,Neutral,Smartphone,Medium,Medium,Medium,Medium,Medium,2. Classic Serif (Traditional fonts with decor...,"2. Medium (14-16px - Standard size, balanced)","Warm Earthy (Browns, beiges, warm oranges - na..."
1,The Loyal Customer (I stick with brands and st...,Happy,Smartphone,Medium,High,Medium,Low,Low,"3. Rounded Friendly (Soft, approachable fonts ...",4. Extra Large (18+px - Maximum readability),"Minimalist Black & White (Clean, high contrast..."
2,"The Minimalist (I want simple, efficient shopp...",Happy,Smartphone,High,Medium,Medium,Medium,Medium,"3. Rounded Friendly (Soft, approachable fonts ...","1. Small (12-14px - Compact, more content visi...","Minimalist Black & White (Clean, high contrast..."
3,The Researcher (I thoroughly research products...,Bored,Smartphone,Low,High,High,Low,Low,"3. Rounded Friendly (Soft, approachable fonts ...",4. Extra Large (18+px - Maximum readability),"Vibrant Bold (Bright, energetic colors - excit..."
4,The Impulsive Buyer (I make quick decisions ba...,Neutral,Smartphone,High,High,Low,Low,High,"4. Bold Impact (Strong, attention-grabbing fonts)","1. Small (12-14px - Compact, more content visi...","Minimalist Black & White (Clean, high contrast..."


## Run Chi-Square Validation


In [9]:
results = run_pairwise_validation(df, predictors, ui_elements)
display(results.head(10))


,Predictor,UI_Element,ChiSquare,DOF,P_Value,Cramers_V,Effect_Size,Sample_Size,Adjusted_P,Significant,Significant_Nominal,Adjusted_P_Per_Predictor,Significant_Per_Predictor
0,current_mood,button_style_pref,65.661972,35,0.001286,0.256246,Medium,200,0.421868,False,True,0.052734,False
1,current_mood,mobile_sticky_header,12.482647,7,0.085763,0.249826,Medium,200,0.615167,False,False,0.439534,False
2,current_mood,desktop_search_visibility,23.865906,14,0.047560,0.244264,Medium,200,0.615167,False,True,0.439534,False
3,Neuroticism_Level,button_style_pref,23.188179,10,0.010073,0.240771,Medium,200,0.589207,False,True,0.412989,False
4,current_mood,desktop_persistent_filters,11.540282,7,0.116727,0.240211,Medium,200,0.615167,False,False,0.470361,False
5,current_mood,mobile_product_card,45.844724,28,0.018084,0.239387,Medium,200,0.615167,False,True,0.370724,False
6,current_mood,desktop_category_display,33.650420,21,0.039484,0.236821,Medium,200,0.615167,False,True,0.439534,False
7,current_mood,desktop_image_text_ratio,22.428104,14,0.070235,0.236792,Medium,200,0.615167,False,False,0.439534,False
8,current_mood,mobile_filter_location,21.826282,14,0.082255,0.233593,Medium,200,0.615167,False,False,0.439534,False
9,primary_device,color_theme_pref,10.524817,5,0.061659,0.229399,Medium,200,0.615167,False,False,0.548632,False


## Export Full Results and Evidence Levels


In [10]:
export_paths = export_statistical_outputs(results, TABLES, top_n=30)
results = summarize_evidence_statistics(results, top_n=30)["enriched_results"]

print("Exported files:")
for name, file_path in export_paths.items():
    print(f"- {name}: {file_path}")

display(results[["Predictor", "UI_Element", "Raw_P", "Adjusted_P", "Adjusted_P_Per_Predictor", "Cramers_V", "Evidence_Score", "Evidence_Level"]].head(10))


INFO: Exported statistical outputs to /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/Statistical_Validation/tables


Exported files:
- csv: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/Statistical_Validation/tables/exploratory_evidence.csv
- xlsx: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/Statistical_Validation/tables/exploratory_evidence.xlsx
- top_evidence_xlsx: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/Statistical_Validation/tables/top_evidence.xlsx
- top30_cramers_v_csv: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/Statistical_Validation/tables/top30_cramers_v.csv
- top30_evidence_score_csv: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/Statistical_Validation/tables/top30_evidence_score.csv


,Predictor,UI_Element,Raw_P,Adjusted_P,Adjusted_P_Per_Predictor,Cramers_V,Evidence_Score,Evidence_Level
0,current_mood,button_style_pref,0.001286,0.421868,0.052734,0.256246,1.000000,Moderate Statistical Evidence
1,current_mood,mobile_sticky_header,0.085763,0.615167,0.439534,0.249826,0.843072,Exploratory Evidence
2,current_mood,desktop_search_visibility,0.047560,0.615167,0.439534,0.244264,0.848940,Moderate Statistical Evidence
3,Neuroticism_Level,button_style_pref,0.010073,0.589207,0.412989,0.240771,0.865523,Moderate Statistical Evidence
4,current_mood,desktop_persistent_filters,0.116727,0.615167,0.470361,0.240211,0.809303,Not Classified
5,current_mood,mobile_product_card,0.018084,0.615167,0.370724,0.239387,0.873640,Moderate Statistical Evidence
6,current_mood,desktop_category_display,0.039484,0.615167,0.439534,0.236821,0.841479,Moderate Statistical Evidence
7,current_mood,desktop_image_text_ratio,0.070235,0.615167,0.439534,0.236792,0.830498,Exploratory Evidence
8,current_mood,mobile_filter_location,0.082255,0.615167,0.439534,0.233593,0.821780,Exploratory Evidence
9,primary_device,color_theme_pref,0.061659,0.615167,0.548632,0.229399,0.790014,Exploratory Evidence


## Evidence Level Counts


In [11]:
print(f"Strong statistical evidence: {results['Is_Strong_Evidence'].sum()}")
print(f"Moderate statistical evidence: {results['Is_Moderate_Evidence'].sum()}")
print(f"Exploratory evidence: {results['Is_Exploratory_Evidence'].sum()}")
print(f"Nominally significant (Raw p < 0.05): {results['Significant_Nominal'].sum()}")
print(f"Global FDR significant: {results['Significant'].sum()}")
print(f"Per-predictor FDR significant: {results['Significant_Per_Predictor'].sum()}")

display(results[results["Is_Moderate_Evidence"]].head(10))


Strong statistical evidence: 0
Moderate statistical evidence: 13
Exploratory evidence: 51
Nominally significant (Raw p < 0.05): 19
Global FDR significant: 0
Per-predictor FDR significant: 0


,Predictor,UI_Element,ChiSquare,DOF,Raw_P,Adjusted_P,Adjusted_P_Per_Predictor,Cramers_V,Effect_Size,Sample_Size,Significant,Significant_Nominal,Significant_Per_Predictor,Evidence_Score,Evidence_Level,Is_Strong_Evidence,Is_Moderate_Evidence,Is_Exploratory_Evidence,P_Value
0,current_mood,button_style_pref,65.661972,35,0.001286,0.421868,0.052734,0.256246,Medium,200,False,True,False,1.000000,Moderate Statistical Evidence,False,True,True,0.001286
2,current_mood,desktop_search_visibility,23.865906,14,0.047560,0.615167,0.439534,0.244264,Medium,200,False,True,False,0.848940,Moderate Statistical Evidence,False,True,True,0.047560
3,Neuroticism_Level,button_style_pref,23.188179,10,0.010073,0.589207,0.412989,0.240771,Medium,200,False,True,False,0.865523,Moderate Statistical Evidence,False,True,True,0.010073
5,current_mood,mobile_product_card,45.844724,28,0.018084,0.615167,0.370724,0.239387,Medium,200,False,True,False,0.873640,Moderate Statistical Evidence,False,True,True,0.018084
6,current_mood,desktop_category_display,33.650420,21,0.039484,0.615167,0.439534,0.236821,Medium,200,False,True,False,0.841479,Moderate Statistical Evidence,False,True,True,0.039484
10,primary_persona,recommendation_type,31.391069,15,0.007784,0.589207,0.159581,0.228732,Medium,200,False,True,False,0.926902,Moderate Statistical Evidence,False,True,True,0.007784
15,primary_persona,color_theme_pref,48.452763,25,0.003291,0.539646,0.134912,0.220120,Medium,200,False,True,False,0.924066,Moderate Statistical Evidence,False,True,True,0.003291
16,Agreeableness_Level,button_style_pref,19.223581,10,0.037513,0.615167,0.719228,0.219224,Medium,200,False,True,False,0.732451,Moderate Statistical Evidence,False,True,True,0.037513
18,primary_device,mobile_grid_pref,9.585976,3,0.022434,0.615167,0.548632,0.218929,Medium,200,False,True,False,0.789432,Moderate Statistical Evidence,False,True,True,0.022434
19,primary_persona,mobile_price_display,18.846122,10,0.042261,0.615167,0.577571,0.217061,Medium,200,False,True,False,0.770959,Moderate Statistical Evidence,False,True,True,0.042261


## Generate Summary Report


In [12]:
summary_path = generate_statistical_summary(
    results,
    REPORTS / "statistical_summary.md",
)
print(f"Summary saved to: {summary_path}")


Summary saved to: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/Statistical_Validation/statistical_summary.md


## Visualizations


In [13]:
plot_association_heatmap(
    results,
    "Adjusted_P",
    HEATMAPS / "adjusted_p_heatmap.png",
    title="Adjusted P-Values (FDR)",
    cmap="YlOrRd_r",
    fmt=".3f",
)
plot_association_heatmap(
    results,
    "Cramers_V",
    HEATMAPS / "cramers_v_heatmap.png",
    title="Cramér's V Effect Sizes",
    cmap="Blues",
    fmt=".2f",
)
plot_top_relationships(
    results,
    FIGURES / "top_30_relationships.png",
    top_n=30,
)


PosixPath('/Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/Statistical_Validation/figures/top_30_relationships.png')

## Final Summary


In [14]:
summary = summarize_evidence_statistics(results, top_n=30)
strongest_v = summary["strongest_cramers_v"]
strongest_score = summary["strongest_evidence_score"]

print(f"Total Tests: {summary['total_tests']}")
print(f"Nominally Significant (Raw p < 0.05): {summary['significant_nominal']}")
print(f"Significant after Global FDR: {summary['significant_global_fdr']}")
print(f"Significant after Predictor-level FDR: {summary['significant_per_predictor_fdr']}")
print(f"Medium Effect Sizes: {summary['medium_effect_sizes']}")
print(f"Large Effect Sizes: {summary['large_effect_sizes']}")
print(f"Strong Evidence: {summary['strong_evidence_count']}")
print(f"Moderate Evidence: {summary['moderate_evidence_count']}")
print(f"Exploratory Evidence: {summary['exploratory_evidence_count']}")
print(
    "Strongest by Cramer's V: "
    f"{strongest_v['Predictor']} × {strongest_v['UI_Element']} "
    f"(V={strongest_v['Cramers_V']:.3f}, Raw p={strongest_v['Raw_P']:.4f})"
)
print(
    "Strongest by Evidence Score: "
    f"{strongest_score['Predictor']} × {strongest_score['UI_Element']} "
    f"(Score={strongest_score['Evidence_Score']:.3f}, V={strongest_score['Cramers_V']:.3f})"
)
print(f"Average Cramer's V: {summary['average_cramers_v']:.3f}")
print(f"Average Evidence Score: {summary['average_evidence_score']:.3f}")
print("Export Locations:")
print(f"- {TABLES / 'statistical_results.xlsx'}")
print(f"- {TABLES / 'strong_evidence.xlsx'}")
print(f"- {TABLES / 'moderate_evidence.xlsx'}")
print(f"- {TABLES / 'exploratory_evidence.xlsx'}")
print(f"- {TABLES / 'top_evidence.xlsx'}")
print(f"- {REPORTS / 'statistical_summary.md'}")


Total Tests: 328
Nominally Significant (Raw p < 0.05): 19
Significant after Global FDR: 0
Significant after Predictor-level FDR: 0
Medium Effect Sizes: 31
Large Effect Sizes: 0
Strong Evidence: 0
Moderate Evidence: 13
Exploratory Evidence: 51
Strongest by Cramer's V: current_mood × button_style_pref (V=0.256, Raw p=0.0013)
Strongest by Evidence Score: current_mood × button_style_pref (Score=1.000, V=0.256)
Average Cramer's V: 0.133
Average Evidence Score: 0.452
Export Locations:
- /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/Statistical_Validation/tables/statistical_results.xlsx
- /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/Statistical_Validation/tables/strong_evidence.xlsx
- /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/Statistical_Validation/tables/moderate_evidence.xlsx
- /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/Statistical_Validation/tables/exploratory_evidence.xlsx
- /Users/mariam/Downl